In [0]:
%run "../../commons/commons_imports"

In [0]:
df_estado_raw = read(
    base_path=RAW_PATH,
    table_name=TS_ESTADO,
    recursive_by_year=True
)

In [0]:
df_estado_bronze = (
    df_estado_raw

    # ============================================
    # Conversão de tipos
    # ============================================
    .withColumn("NU_ANO_AVALIACAO", col("NU_ANO_AVALIACAO").cast("int"))
    .withColumn("CO_UF", col("CO_UF").cast("int"))
    .withColumn("TP_SERIE", col("TP_SERIE").cast("int"))
    .withColumn("ID_TIPO_REDE", col("ID_TIPO_REDE").cast("int"))

    .withColumn("PC_ALUNO_ALFABETIZADO", col("PC_ALUNO_ALFABETIZADO").cast("double"))
    .withColumn("VL_MEDIA_LP", col("VL_MEDIA_LP").cast("double"))

    .withColumn("PC_ALUNO_NIVEL_0_LP", col("PC_ALUNO_NIVEL_0_LP").cast("double"))
    .withColumn("PC_ALUNO_NIVEL_1_LP", col("PC_ALUNO_NIVEL_1_LP").cast("double"))
    .withColumn("PC_ALUNO_NIVEL_2_LP", col("PC_ALUNO_NIVEL_2_LP").cast("double"))
    .withColumn("PC_ALUNO_NIVEL_3_LP", col("PC_ALUNO_NIVEL_3_LP").cast("double"))
    .withColumn("PC_ALUNO_NIVEL_4_LP", col("PC_ALUNO_NIVEL_4_LP").cast("double"))
    .withColumn("PC_ALUNO_NIVEL_5_LP", col("PC_ALUNO_NIVEL_5_LP").cast("double"))
    .withColumn("PC_ALUNO_NIVEL_6_LP", col("PC_ALUNO_NIVEL_6_LP").cast("double"))
    .withColumn("PC_ALUNO_NIVEL_7_LP", col("PC_ALUNO_NIVEL_7_LP").cast("double"))
    .withColumn("PC_ALUNO_NIVEL_8_LP", col("PC_ALUNO_NIVEL_8_LP").cast("double"))

    # ============================================
    # Padronização
    # ============================================
    .withColumn("SG_UF", upper(trim(col("SG_UF"))))

    # ============================================
    # Colunas descritivas
    # ============================================
    .withColumn(
        "DS_TIPO_REDE",
        when(col("ID_TIPO_REDE") == 2, "Estadual")
        .when(col("ID_TIPO_REDE") == 3, "Municipal")
        .when(col("ID_TIPO_REDE") == 4, "Privada")
        .when(col("ID_TIPO_REDE") == 5, "Total")
    )

    # ============================================
    # Colunas técnicas
    # ============================================
    .withColumn("DT_PROCESSAMENTO", current_date())
    .withColumn("TS_PROCESSAMENTO", current_timestamp())

    # ============================================
    # Remove duplicados
    # ============================================
    .dropDuplicates([
        "NU_ANO_AVALIACAO",
        "CO_UF",
        "TP_SERIE",
        "ID_TIPO_REDE"
    ])
)

In [0]:
'''
ID único para todos as linhas
'''
df_estado_bronze = df_estado_bronze.withColumn(
    "SK_ESTADO",
    sha2(
        concat_ws(
            "|",
            col("NU_ANO_AVALIACAO"),
            col("CO_UF"),
            col("TP_SERIE"),
            col("ID_TIPO_REDE")
        ),
        256
    )
)

In [0]:
df_estado_bronze_selected = df_estado_bronze.select(
    # ============================
    # Chave técnica
    # ============================
    "SK_ESTADO",

    # ============================
    # Chaves de negócio
    # ============================
    "NU_ANO_AVALIACAO",
    "ANO_REFERENCIA",
    "CO_UF",
    "SG_UF",
    "TP_SERIE",
    "ID_TIPO_REDE",

    # ============================
    # Métricas
    # ============================
    "PC_ALUNO_ALFABETIZADO",
    "VL_MEDIA_LP",
    "PC_ALUNO_NIVEL_0_LP",
    "PC_ALUNO_NIVEL_1_LP",
    "PC_ALUNO_NIVEL_2_LP",
    "PC_ALUNO_NIVEL_3_LP",
    "PC_ALUNO_NIVEL_4_LP",
    "PC_ALUNO_NIVEL_5_LP",
    "PC_ALUNO_NIVEL_6_LP",
    "PC_ALUNO_NIVEL_7_LP",
    "PC_ALUNO_NIVEL_8_LP",

    # ============================
    # Colunas descritivas
    # ============================
    "DS_TIPO_REDE",

    # ============================
    # Colunas técnicas
    # ============================
    "DT_PROCESSAMENTO",
    "TS_PROCESSAMENTO"
)

In [0]:
write_delta(
    df=df_estado_bronze_selected,
    base_path=BRONZE_PATH,
    table_name=TS_ESTADO,
    merge_keys=["SK_ESTADO"]
)